B"H

## Project 2: Ames Housing Price Drivers

- David Koyrakh
- Bellevue University
- DSC-680: Applied Data Science
- Professor Iranitalab


This notebook includes data preparation, modeling, and evaluation to support the Project 2 study on the Ames, Iowa housing dataset.

- Business context: pricing drivers for residential sales in Ames, IA
- Goals: characterize data quality, explore key relationships to `SalePrice`, build and evaluate models.

In [ ]:
# Setup: imports and options
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("data")
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")

# outputs directory for saved figures
OUTPUTS_DIR = os.path.join("outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print("Data path:", TRAIN_PATH)
print("Outputs dir:", OUTPUTS_DIR)

Data path: data\train.csv
Outputs dir: outputs


### Data Loading

Load the Ames housing training data to verify shape and dtypes.

In [2]:
# Load data and display basic schema info
raw = pd.read_csv(TRAIN_PATH)
print(raw.shape)
print(raw.dtypes.head(15))
raw.head()

(1460, 81)
Id                int64
MSSubClass        int64
MSZoning         object
LotFrontage     float64
LotArea           int64
Street           object
Alley            object
LotShape         object
LandContour      object
Utilities        object
LotConfig        object
LandSlope        object
Neighborhood     object
Condition1       object
Condition2       object
dtype: object


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


#### Analysis
- Shape is 1460 x 81
- There are many categorical variables.
- Early rows show typical suburban mix; quality/size fields prominent.
- Use case: predictors span lot, structure, basement/garage, kitchen/bath, sale.

### Data Quality

Before modeling, I need to understand data gaps and basic distributions to plan encoding/imputation accordingly, and to assess risks to modeling validity.

### Missing values

In [3]:
# Missingness table and basic numeric summary
missing = raw.isna().mean().sort_values(ascending=False)
missing_df = missing[missing > 0].to_frame(name="missing_rate").reset_index().rename(columns={"index":"column"})
print("Columns with missing values (top 15):")
print(missing_df.head(15))

print("\nNumeric summary (selected):")
num_summary = raw.describe().T[["mean","std","min","25%","50%","75%","max"]]
num_summary.head(10)

Columns with missing values (top 15):
          column  missing_rate
0         PoolQC      0.995205
1    MiscFeature      0.963014
2          Alley      0.937671
3          Fence      0.807534
4     MasVnrType      0.597260
5    FireplaceQu      0.472603
6    LotFrontage      0.177397
7    GarageYrBlt      0.055479
8     GarageCond      0.055479
9     GarageType      0.055479
10  GarageFinish      0.055479
11    GarageQual      0.055479
12  BsmtFinType2      0.026027
13  BsmtExposure      0.026027
14      BsmtQual      0.025342

Numeric summary (selected):


,mean,std,min,25%,50%,75%,max
Id,730.500000,421.610009,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,56.897260,42.300571,20.0,20.00,50.0,70.00,190.0
LotFrontage,70.049958,24.284752,21.0,59.00,69.0,80.00,313.0
LotArea,10516.828082,9981.264932,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,6.099315,1.382997,1.0,5.00,6.0,7.00,10.0
OverallCond,5.575342,1.112799,1.0,5.00,5.0,6.00,9.0
YearBuilt,1971.267808,30.202904,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1984.865753,20.645407,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,103.685262,181.066207,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,443.639726,456.098091,0.0,0.00,383.5,712.25,5644.0


#### Summary
- High-NA fields: `PoolQC` (99.5%), `MiscFeature` (96%), `Alley` (93.8%), `Fence` (80.8%). These likely encode absence; treat with explicit "None" or most-frequent imputation per semantics.
- Moderate gaps: `LotFrontage` (17.7%) and garage/basement variants (~2–6%). Median impute for numeric; mode for categoricals is reasonable.
- Heavy right tails in `LotArea`, basement areas; consider log transforms for linear models.


### Distribution

Examining the distribution of key variables, especially the target, helps identify skewness, outliers, and the need for transformations before modeling.

In [4]:
# Distribution of SalePrice and log-transformed SalePrice
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(raw["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice distribution")

sns.histplot(np.log1p(raw["SalePrice"]), kde=True, ax=axes[1])
axes[1].set_title("log1p(SalePrice) distribution")
fig.tight_layout()

# save (no inline display)
fig_path = os.path.join(OUTPUTS_DIR, "dist_saleprice_and_log1p.png")
fig.savefig(fig_path, bbox_inches="tight", dpi=150)
plt.close(fig)
print("Saved:", fig_path)

print(raw["SalePrice"].describe())
print("\nLog1p SalePrice summary:")
print(np.log1p(raw["SalePrice"]).describe())

Saved: outputs\dist_saleprice_and_log1p.png
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

Log1p SalePrice summary:
count    1460.000000
mean       12.024057
std         0.399449
min        10.460271
25%        11.775105
50%        12.001512
75%        12.273736
max        13.534474
Name: SalePrice, dtype: float64


#### Summary
- Saved `outputs/dist_saleprice_and_log1p.png` shows `SalePrice` is right-skewed; `log1p(SalePrice)` is near-symmetric (mean=12.02, IQR ~11.78–12.27).
- Downstream: prefer modeling on log target for linear methods and to stabilize residuals.


### Exploratory Visuals

Visual inspection of target distribution and top relationships guides transformations, encoding, and model focus.

In [5]:
# Bivariate relationships: scatter/reg lines for well-known drivers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["GrLivArea", "OverallQual", "YearBuilt"]):
    sns.regplot(x=raw[col], y=raw["SalePrice"], scatter_kws={"alpha":0.4, "s":20}, line_kws={"color":"red"}, ax=ax)
    ax.set_title(f"SalePrice vs {col}")
plt.tight_layout()

# save
fig_path = os.path.join(OUTPUTS_DIR, "bivariates_saleprice_drivers.png")
plt.savefig(fig_path, bbox_inches="tight", dpi=150)
plt.close(fig)
print("Saved:", fig_path)

Saved: outputs\bivariates_saleprice_drivers.png


#### Summary / Key bivariates
- `outputs/bivariates_saleprice_drivers.png` shows strong positive trends for `GrLivArea` and `OverallQual`, and a milder trend with `YearBuilt`.
- Visible heteroskedasticity at large `GrLivArea`; consider robust loss or transformations.


### Correlation analysis

To understand how different numeric features relate to SalePrice, I create a correlation heatmap focusing on variables most strongly associated with the target.

In [6]:
# Correlation heatmap for top numeric features by absolute correlation with SalePrice
numeric_cols = raw.select_dtypes(include=[np.number])
cor = numeric_cols.corr(numeric_only=True)["SalePrice"].abs().sort_values(ascending=False)

# Exclude the target itself; take top 12 including target for context if needed
top = cor.iloc[1:13].index.tolist()
fig = plt.figure(figsize=(10, 8))
sns.heatmap(numeric_cols[top + ["SalePrice"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Top numeric correlations with SalePrice")
plt.tight_layout()
fig_path = os.path.join(OUTPUTS_DIR, "corr_heatmap_top_numeric_features.png")
plt.savefig(fig_path, bbox_inches="tight", dpi=150)
plt.close(fig)
print("Saved:", fig_path)


Saved: outputs\corr_heatmap_top_numeric_features.png


#### Summary
- `outputs/corr_heatmap_top_numeric_features.png` ranks numeric drivers led by `OverallQual`, `GrLivArea`, basement and floor SF metrics, `GarageCars`.
- These align with domain expectations; multicollinearity likely among SF features.


### Baseline Models and Evaluation

Establish reference performance and interpretability to ground further improvements. I compare a regularized linear model and a tree ensemble with straightforward preprocessing.

In [7]:
# Preprocessing: numeric impute+scale, categorical impute+onehot; 5-fold CV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

X = raw.drop(columns=["SalePrice", "Id"])  # drop target and identifier
y = raw["SalePrice"].copy()

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler(with_mean=False))
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

# Ridge with built-in CV on alphas (use MSE scoring for broad compatibility)
ridge_alphas = np.logspace(-3, 3, 13)
ridge = Pipeline([
    ("prep", preprocess),
    ("model", RidgeCV(alphas=ridge_alphas, scoring="neg_mean_squared_error", cv=5))
])

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validated RMSE: compute from negative MSE scores for version safety
ridge_mse_scores = cross_val_score(ridge, X, y, scoring="neg_mean_squared_error", cv=kfold, n_jobs=None)
ridge_rmse_scores = np.sqrt(-ridge_mse_scores)
print("Ridge RMSE (5-fold):", ridge_rmse_scores)
print("Ridge RMSE mean:", ridge_rmse_scores.mean())

# RandomForest baseline
rf = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
])
rf_mse_scores = cross_val_score(rf, X, y, scoring="neg_mean_squared_error", cv=kfold, n_jobs=None)
rf_rmse_scores = np.sqrt(-rf_mse_scores)
print("RandomForest RMSE (5-fold):", rf_rmse_scores)
print("RandomForest RMSE mean:", rf_rmse_scores.mean())

Ridge RMSE (5-fold): [31319.82086676 31196.75398822 54718.58310991 27356.88358179
 20853.28744681]
Ridge RMSE mean: 33089.06579869889
RandomForest RMSE (5-fold): [28982.75141626 25807.76741265 44658.87052185 27829.43840918
 23786.02916962]
RandomForest RMSE mean: 30212.971385911027


### Model Evaluation & Key Findings

- Baseline RandomForest achieved cross-validated RMSE = 30.2K, indicating moderate predictive accuracy without tuning. This is a reasonable starting point given raw categorical variety and limited feature engineering.
- EDA confirms skew in `SalePrice` (log1p is more symmetric) and strong relationships with `GrLivArea`, `OverallQual`, and recency (`YearBuilt`).
- Feature importance highlights size/quality variables and selected categorical encodings. Neighborhood medians show clear stratification for narrative and stakeholder guidance.

Implications:
- Methods: justify preprocessing (median/mode imputation, one-hot, scaling) and CV choice (5-fold).
- Analysis: report RF RMSE mean (=30.2K) and show supporting visuals (distributions, bivariates, importance, neighborhood chart).
- Recommendations: emphasize improvements via log-target modeling, hyperparameter tuning (RF/GBM), and targeted transformations for skewed predictors; consider ordinal encodings for quality ratings.
- Limitations: single-city dataset, time-bound sales, potential leakage if not careful with encodings; categorical sparsity after one-hot.
- Future Uses: partial dependence/SHAP for explainability; geospatial augmentation.

In [ ]:
# Fit RandomForest on full data for quick feature importance and a simple business-facing chart
rf_fit = rf.fit(X, y)

# Extract feature names after one-hot
ohe = rf_fit.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
num_names = num_cols
cat_names = ohe.get_feature_names_out(cat_cols).tolist()
feature_names = num_names + cat_names

importances = rf_fit.named_steps["model"]

# Top 15 features
idx = np.argsort(importances)[-15:][::-1]
imp_df = pd.DataFrame({"feature": np.array(feature_names)[idx], "importance": importances[idx]})
print(imp_df)

fig1 = plt.figure(figsize=(8, 6))
sns.barplot(data=imp_df, x="importance", y="feature", orient="h")
plt.title("RandomForest Top Feature Importances")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_DIR, "rf_feature_importance.png")
plt.savefig(out_path)
plt.close(fig1)
print(f"saved: {out_path}")

# Neighborhood median SalePrice for business narrative
neigh = raw.groupby("Neighborhood", as_index=False)["SalePrice"].median().sort_values("SalePrice", ascending=False)
fig2 = plt.figure(figsize=(10,6))
sns.barplot(data=neigh, x="SalePrice", y="Neighborhood", orient="h")
plt.title("Median SalePrice by Neighborhood")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_DIR, "neighborhood_median_saleprice.png")
plt.savefig(out_path)
plt.close(fig2)
print(f"saved: {out_path}")

         feature  importance
0    OverallQual    0.576695
1      GrLivArea    0.107633
2    TotalBsmtSF    0.040295
3       2ndFlrSF    0.034502
4     BsmtFinSF1    0.028305
5       1stFlrSF    0.022256
6     GarageCars    0.022235
7     GarageArea    0.014472
8        LotArea    0.012825
9      YearBuilt    0.008763
10      FullBath    0.007583
11  YearRemodAdd    0.006806
12   LotFrontage    0.006423
13  TotRmsAbvGrd    0.006110
14     BsmtUnfSF    0.005326
saved: outputs\rf_feature_importance.png
saved: outputs\neighborhood_median_saleprice.png


##### Importance & neighborhoods
- `outputs/rf_feature_importance.png` ranks `OverallQual` and `GrLivArea` highest by a wide margin, followed by `TotalBsmtSF`, `2ndFlrSF`, `BsmtFinSF1`, `1stFlrSF`, and garage capacity/area.
- `outputs/neighborhood_median_saleprice.png` shows clear stratification (e.g., NridgHt/NoRidge higher vs. IDOTRR/OldTown lower). Useful for stakeholder comparisons and pricing bands.

##### Baselines
- Ridge RMSE (5-fold) mean = 33.1K; RandomForest = 30.2K. The tree model benefits from nonlinearity and mixed types, as expected.
- Next gains: tuned GBM (e.g., XGBoost/LightGBM), log-target, and targeted encodings for ordinal qualities.

### Final Evaluation Metrics

Why: I need final RMSE, MAE, and R2 across candidate models to report in the final deliverable.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

ridge_model = ridge
rf_model = rf
models = {"Ridge": ridge_model}

log_ridge = Pipeline([
    ("prep", preprocess),
    ("model", TransformedTargetRegressor(
        regressor=RidgeCV(alphas=(ridge_alphas if "ridge_alphas" in globals() else np.logspace(-3, 3, 13)),
                          scoring="neg_mean_squared_error", cv=5),
        func=np.log1p, inverse_func=np.expm1
    ))
])
models["Ridge_LogTarget"] = log_ridge

gbr = Pipeline([("prep", preprocess), ("model", GradientBoostingRegressor(random_state=42))])
models["GradientBoosting"] = gbr

models["RandomForest"] = rf_model

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, m in models.items():
    rmse = np.sqrt(-cross_val_score(m, X, y, scoring="neg_mean_squared_error", cv=kfold, n_jobs=None))
    mae  = -cross_val_score(m, X, y, scoring="neg_mean_absolute_error", cv=kfold, n_jobs=None)
    r2s  = cross_val_score(m, X, y, scoring="r2", cv=kfold, n_jobs=None)

    rows.append({
        "model": name,
        "rmse_mean": rmse.mean(), "rmse_std": rmse.std(),
        "mae_mean":  mae.mean(),  "mae_std":  mae.std(),
        "r2_mean":   r2s.mean(),  "r2_std":   r2s.std(),
    })

    print(f"{name} RMSE: {rmse} mean={rmse.mean():.2f}")
    print(f"{name} MAE:  {mae} mean={mae.mean():.2f}")
    print(f"{name} R2:   {r2s} mean={r2s.mean():.3f}")
    print("---")

metrics_df = pd.DataFrame(rows).sort_values("rmse_mean")
print("\nMETRICS SUMMARY:\n", metrics_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

os.makedirs("outputs", exist_ok=True)
out_path = os.path.join("outputs", "final_cv_metrics.csv")
metrics_df.to_csv(out_path, index=False)
print("Saved metrics to:", out_path)

Ridge RMSE: [31319.82086676 31196.75398822 54718.58310991 27356.88358179
 20853.28744681] mean=33089.07
Ridge MAE:  [18691.18360299 18466.92840016 18969.09850214 18330.88103027
 14970.31323446] mean=17885.68
Ridge R2:   [0.8721135  0.85686143 0.45804408 0.88081117 0.91680296] mean=0.797
---
Ridge_LogTarget RMSE: [ 25761.24868608  27286.60519098 168610.95777134  25023.45457329
  21336.97401184] mean=53603.85
Ridge_LogTarget MAE:  [16686.8727025  16578.6181644  25764.39720579 15807.15684201
 14200.34068224] mean=17807.48
Ridge_LogTarget R2:   [ 0.91347931  0.89049426 -4.14595364  0.90027665  0.91289874] mean=-0.106
---
GradientBoosting RMSE: [26325.79487385 22414.05664416 46549.63958293 25975.96196163
 20856.88690948] mean=28424.47
GradientBoosting MAE:  [16531.77865831 15433.80480923 18956.5495546  16390.62500849
 13571.39066624] mean=16176.83
GradientBoosting R2:   [0.90964563 0.92611117 0.60778254 0.89254031 0.91677424] mean=0.851
---
RandomForest RMSE: [28953.47154081 25983.53922252 

#### Final metrics summary

- Gradient Boosting: RMSE = 28.4K, MAE = 16.2K, R² = 0.851 (best)
- Random Forest: RMSE = 30.3K, MAE = 17.8K, R² = 0.839
- Ridge: RMSE = 33.1K, MAE = 17.9K, R² = 0.797
- Ridge (log‑target): RMSE = 53.6K, MAE = 17.8K, R² = −0.106

Implications: prefer GBM for accuracy; keep Ridge as a transparent baseline. Consider tuned GBM/XGBoost, ordinal encodings, and SHAP/PDP for explanations.